# Heart Attack Detection Using Machine Learning

This notebook shows a simple machine learning workflow for detecting heart attack risk. It uses a synthetic dataset with common health-related features, trains two classification models, and compares their performance.

**Required date:** 2023-12-13

> Note: This is an educational project. The data is synthetic and the model must not be used for real medical decisions.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42

## 2. Create a Simple Dataset

For this basic project, we generate a synthetic dataset that looks like a heart-health dataset. The target column is `heart_attack_risk`, where `1` means higher risk and `0` means lower risk.

In [ ]:
feature_names = [
    "age",
    "cholesterol",
    "resting_bp",
    "max_heart_rate",
    "exercise_angina",
    "blood_sugar",
    "oldpeak",
    "chest_pain_type"
]

X, y = make_classification(
    n_samples=500,
    n_features=len(feature_names),
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    weights=[0.55, 0.45],
    class_sep=1.1,
    random_state=RANDOM_STATE
)

df = pd.DataFrame(X, columns=feature_names)

# Convert generated values into easier-to-read clinical-style ranges.
df["age"] = np.clip((df["age"] * 8 + 52).round(), 29, 78)
df["cholesterol"] = np.clip((df["cholesterol"] * 35 + 220).round(), 120, 360)
df["resting_bp"] = np.clip((df["resting_bp"] * 18 + 130).round(), 85, 200)
df["max_heart_rate"] = np.clip((df["max_heart_rate"] * 22 + 150).round(), 70, 205)
df["exercise_angina"] = (df["exercise_angina"] > df["exercise_angina"].median()).astype(int)
df["blood_sugar"] = np.clip((df["blood_sugar"] * 25 + 105).round(), 70, 220)
df["oldpeak"] = np.clip((df["oldpeak"] * 0.9 + 1.2).round(1), 0, 6)
df["chest_pain_type"] = pd.qcut(df["chest_pain_type"], q=4, labels=[0, 1, 2, 3]).astype(int)
df["heart_attack_risk"] = y

df.head()

## 3. Quick Data Check

In [ ]:
print(df.shape)
display(df.describe())
display(df["heart_attack_risk"].value_counts().rename(index={0: "low_risk", 1: "high_risk"}))

## 4. Split Data

In [ ]:
X = df.drop(columns="heart_attack_risk")
y = df["heart_attack_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 5. Train Models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=120,
        max_depth=6,
        random_state=RANDOM_STATE
    )
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    results.append({"model": name, "accuracy": accuracy})
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values(by="accuracy", ascending=False)
results_df

## 6. Evaluate the Best Model

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]
y_pred = best_model.predict(X_test)

print("Best model:", best_model_name)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["Low risk", "High risk"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
display_labels = ["Low risk", "High risk"]

ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels).plot(cmap="Blues")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.show()

## 7. Try a New Prediction

In [ ]:
sample_patient = pd.DataFrame([
    {
        "age": 58,
        "cholesterol": 245,
        "resting_bp": 142,
        "max_heart_rate": 118,
        "exercise_angina": 1,
        "blood_sugar": 135,
        "oldpeak": 2.4,
        "chest_pain_type": 2
    }
])

prediction = best_model.predict(sample_patient)[0]
probability = best_model.predict_proba(sample_patient)[0][1]

print("Prediction:", "High risk" if prediction == 1 else "Low risk")
print("Estimated high-risk probability:", round(probability, 3))

## Conclusion

This notebook demonstrates a basic supervised machine learning workflow: create data, split it, train models, evaluate results, and make a prediction for a new patient example. For a real project, the next step would be using a verified medical dataset and validating the model carefully.